# Lecture 03 — Matrices: The Spreadsheet of Mathematics · Laboratory

Read `blog.md` first. The notebook follows: predict → calculate → verify → experiment.

In [ ]:
import numpy as np

rooms = np.array([2.,2.,3.,4.])
area = np.array([800.,1200.,900.,1600.])
prices = np.array([9.,11.,11.5,17.])
X = np.column_stack([rooms, area])
w = np.array([2.,0.005])
b = 1.

print('X shape:', X.shape)
print('loop:', np.array([row @ w + b for row in X]))
print('matrix:', X @ w + b)
assert np.allclose(X @ w + b, prices)

## Shape prediction

Before running, predict the shapes of `(4,2) @ (2,3)` and its result.

In [ ]:
W = np.array([[2.,0.,1.2],[0.005,0.004,0.003]])
biases = np.array([1.,0.,0.6])
Y = X @ W + biases
print('X:', X.shape, 'W:', W.shape, 'Y:', Y.shape)
print(Y)
assert Y.shape == (4,3)
assert np.allclose(Y[:,0], prices)
assert np.allclose(Y[:,2], 0.6 * Y[:,0])

## One entry by hand

Compute house A's tax value by hand before running this cell.

In [ ]:
entry = X[0,0]*W[0,2] + X[0,1]*W[1,2]
print('raw (XW)[0,2] =', entry)
print('with bias =', entry + biases[2])
assert entry == 4.8
assert entry + biases[2] == 5.4

## Explicit matrix multiplication

The library call is just optimized loops. Write the loops yourself.

In [ ]:
def explicit_matmul(A, B):
    n, d = A.shape
    d2, m = B.shape
    assert d == d2
    C = np.zeros((n,m))
    for i in range(n):
        for j in range(m):
            for k in range(d):
                C[i,j] += A[i,k] * B[k,j]
    return C

assert np.allclose(explicit_matmul(X,W), X @ W)
print('explicit multiplication matches NumPy')

## Transpose and gradients

If `error` has one value per example, which shape should the weight gradient have?

In [ ]:
error = X @ w + b - prices
g = X.T @ error
print('X shape:', X.shape)
print('error shape:', error.shape)
print('X.T shape:', X.T.shape)
print('gradient shape:', g.shape)
assert g.shape == w.shape
try:
    X @ error
    raise AssertionError('X @ error should not be the intended vector gradient')
except ValueError:
    print('X @ error fails the shape rule, as expected')

## Non-commutativity

Find a counterexample to `AB = BA`. 

In [ ]:
A = np.array([[1.,1.],[0.,1.]])
B = np.array([[1.,0.],[1.,1.]])
print('AB =\n', A @ B)
print('BA =\n', B @ A)
assert not np.allclose(A @ B, B @ A)

## Feature scaling experiment

Scale both columns to roughly `[0,1]`. Verify that the fitted predictions can remain the same after rescaling the weights.

In [ ]:
X_scaled = X / np.array([4.,1600.])
w_scaled = np.array([2.*4., 0.005*1600.])
print('raw X @ w + b:', X @ w + b)
print('scaled X @ scaled_w + b:', X_scaled @ w_scaled + b)
assert np.allclose(X @ w + b, X_scaled @ w_scaled + b)